# 4.1 Genetic Algorithm for Feature Selection


In [ ]:
import numpy as np
import random
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

if os.path.exists('../data/raw_data.csv'):
    path = '../data/raw_data.csv'
else:
    path = 'data/raw_data.csv'

df = pd.read_csv(path).sample(1000, random_state=42)
df.ffill(inplace=True)
for c in df.select_dtypes(['object']).columns:
    df[c] = df[c].astype('category').cat.codes

X = df.drop(columns=['id', 'Response'], errors='ignore').values
y = df['Response'].values if 'Response' in df.columns else np.random.randint(0, 2, len(df))


## 4.2 GA Logic Implementation


In [ ]:
def fitness_function(chromosome):
    selected_indices = [i for i, bit in enumerate(chromosome) if bit == 1]
    if len(selected_indices) == 0:
        return 0.0
    X_subset = X[:, selected_indices]
    clf = RandomForestClassifier(n_estimators=10, random_state=42)
    scores = cross_val_score(clf, X_subset, y, cv=3)
    return scores.mean()

def run_ga(num_features, generations=10, pop_size=10):
    population = [[random.randint(0, 1) for _ in range(num_features)] for _ in range(pop_size)]
    for gen in range(generations):
        fitnesses = [fitness_function(ind) for ind in population]
        best_idx = np.argmax(fitnesses)
        print(f"Gen {gen+1} | Best Accuracy: {fitnesses[best_idx]:.4f}")
        
        # Simple selection and mutation
        next_gen = [population[best_idx]] # Elitism
        for _ in range(pop_size - 1):
            parent = population[random.choice(np.argsort(fitnesses)[-3:])]
            child = parent.copy()
            mutation_pt = random.randint(0, num_features - 1)
            child[mutation_pt] = 1 - child[mutation_pt]
            next_gen.append(child)
        population = next_gen
    
    return population[0]

features_map = df.drop(columns=['id', 'Response'], errors='ignore').columns
best_chromosome = run_ga(len(features_map))
print("Best Feature Selection:", [features_map[i] for i, b in enumerate(best_chromosome) if b == 1])
